<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity/V2_Validation_%26_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Burst Detection Model Validation and Testing - V2

This notebook evaluates the performance of the **Module 3 Isolation Forest model (v2)** on validation and testing datasets to identify product review bursts.

### Objectives
- Load pre-computed sliding window features and the trained Isolation Forest model.
- Score the validation and testing windows using the model's decision function.
- Determine the optimal anomaly score threshold on the validation split.
- Evaluate the final performance (Recall, Precision, F1-score, and False Positive Rate) on the locked test split.

### Inputs
1. **Features File:** `features_val_test.csv` (contains feature windows for val/test splits)
2. **Combined Splits Data:** `df_combined_split.csv` (contains review-level data to locate synthetic bursts)
3. **Model Artifact:** `isolation_forest_module3_v2.pkl` (trained Isolation Forest classifier)
4. **Feature Columns List:** `module3_feature_cols_v2.pkl`

### Outputs
- **Model Decision Scores:** Appended to the feature dataset.
- **Optimal Threshold Selection:** Analyzed and selected on validation metrics ($F_1$-optimal threshold: `-0.07`).
- **Final Evaluation Metrics:** Final performance summary on the held-out test split.

---

### Key Notes
- **Aggregation Rule:** Predictions are aggregated at the **product level** by taking the *minimum anomaly score* across all windows for a given product.
- **Burst Definition:** A product is defined as a synthetic burst target if it has $\ge 5$ generated (synthetic) reviews in that split.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, joblib

BASE  = "/content/drive/MyDrive"
FINAL = f"{BASE}/Module3_Final"
EXP   = f"{BASE}/module3_burst_experiment"   # where the v2 model lives

# load features
feats = pd.read_csv(f"{FINAL}/features_val_test.csv")

# load v2 model
model    = joblib.load(f"{EXP}/isolation_forest_module3_v2.pkl")
fcols    = joblib.load(f"{EXP}/module3_feature_cols_v2.pkl")

feats['anomaly_score'] = model.decision_function(feats[fcols])
print("Scored. Features:", fcols)
print("Val windows:", (feats['split']=='val').sum(),
      "Test windows:", (feats['split']=='test').sum())

MessageError: Error: credential propagation was unsuccessful

In [ ]:
df = pd.read_csv(f"{FINAL}/df_combined_split.csv")

# burst-bearing products per split (>=5 synthetic reviews)
def burst_products(split):
    s = df[(df['split']==split) & (df['is_generated']==1)]
    counts = s.groupby('product_id').size()
    return set(counts[counts>=5].index), set(s['product_id'].unique())

val_burst, val_targeted = burst_products('val')

# per-product min score on VAL
fv = feats[feats['split']=='val']
prod_min = fv.groupby('product_id')['anomaly_score'].min()

b = prod_min[prod_min.index.isin(val_burst)].values
g = prod_min[~prod_min.index.isin(val_targeted)].values

print(f"VAL: {len(b)} burst products, {len(g)} genuine\n")
print("Threshold │ Recall │ Gen FP │ Precision │  F1")
print("──────────┼────────┼────────┼───────────┼──────")
best = None
for t in [-0.12,-0.10,-0.08,-0.07,-0.06,-0.05,-0.04,-0.03,-0.02]:
    r  = (b < t).mean()
    fp = (g < t).mean()
    tp = r*len(b); FP = fp*len(g)
    p  = tp/(tp+FP) if (tp+FP)>0 else 0
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f"  {t:+.2f}   │ {r*100:5.1f}% │ {fp*100:5.1f}% │  {p*100:5.1f}%  │ {f1*100:4.1f}%")
    if best is None or f1 > best[1]: best = (t, f1)

print(f"\n>>> VAL-selected threshold: {best[0]:+.2f} (F1 {best[1]*100:.1f}%)")

VAL: 310 burst products, 10584 genuine

Threshold │ Recall │ Gen FP │ Precision │  F1
──────────┼────────┼────────┼───────────┼──────
  -0.12   │   0.6% │   0.1% │   12.5%  │  1.2%
  -0.10   │  25.5% │   0.6% │   57.2%  │ 35.3%
  -0.08   │  43.9% │   1.4% │   48.1%  │ 45.9%
  -0.07   │  73.9% │   2.0% │   52.0%  │ 61.1%
  -0.06   │  78.7% │   3.0% │   43.2%  │ 55.8%
  -0.05   │  79.0% │   4.6% │   33.3%  │ 46.9%
  -0.04   │  97.7% │   6.9% │   29.3%  │ 45.1%
  -0.03   │ 100.0% │   9.2% │   24.0%  │ 38.8%
  -0.02   │ 100.0% │  11.4% │   20.4%  │ 33.9%

>>> VAL-selected threshold: -0.07 (F1 61.1%)


In [ ]:
LOCKED = -0.07   # chosen on val

test_burst, test_targeted = burst_products('test')

ft = feats[feats['split']=='test']
prod_min_t = ft.groupby('product_id')['anomaly_score'].min()

b = prod_min_t[prod_min_t.index.isin(test_burst)].values
g = prod_min_t[~prod_min_t.index.isin(test_targeted)].values

TP = (b < LOCKED).sum(); FN = len(b)-TP
FP = (g < LOCKED).sum(); TN = len(g)-FP
recall    = TP/len(b)
precision = TP/(TP+FP) if (TP+FP)>0 else 0
f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0

print("="*52)
print(f"FINAL TEST RESULTS (threshold {LOCKED}, locked on val)")
print("="*52)
print(f"  Burst products:  {len(b)}")
print(f"  Genuine products: {len(g)}")
print(f"  TP={TP}  FN={FN}  FP={FP}  TN={TN}")
print(f"  Recall:    {recall*100:.1f}%")
print(f"  Precision: {precision*100:.1f}%")
print(f"  F1:        {f1*100:.1f}%")
print(f"  FP rate:   {FP/len(g)*100:.1f}%")

FINAL TEST RESULTS (threshold -0.07, locked on val)
  Burst products:  445
  Genuine products: 9913
  TP=313  FN=132  FP=226  TN=9687
  Recall:    70.3%
  Precision: 58.1%
  F1:        63.6%
  FP rate:   2.3%
